# NOARK Force Validation — Multi-location Analysis
Aggregates sweep results across multiple positions.
Each session folder must contain: , , .
Populate  in the pipeline cell, then run all cells.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d


In [ ]:
BASE = '/home/sujith/Documents/NOARK_backbone/csv_data'

# ── Add/remove sessions here ─────────────────────────────────────────────────
SESSIONS = [
    ('center_t1', f'{BASE}/june_22_test'),
    ('center_t2', f'{BASE}/position_1_june_22'),
    ('left_t1',   f'{BASE}/position_2_june_22'),
    ('left_t2',   f'{BASE}/position_2_june_22_t1'),
    ('right_t1',  f'{BASE}/position_3_june_22'),
]

_HOLD = 2.0;  _STABLE = 1.0;  _NSTEPS = 7 * 4;  _HZ = 100
_MAGS = [5.0, 10.0, 15.0, 24.0]

all_summaries = []

for label, sdir in SESSIONS:
    try:
        lc_  = pd.read_csv(f'{sdir}/loadcell.csv', parse_dates=['timestamp']).sort_values('timestamp')
        gui_ = pd.read_csv(f'{sdir}/gui.csv',       parse_dates=['timestamp']).sort_values('timestamp')
        pos_ = pd.read_csv(f'{sdir}/position.csv')
        pos_x, pos_z = float(pos_['x'].iloc[0]), float(pos_['z'].iloc[0])
    except Exception as e:
        print(f'[{label}] skip — {e}'); continue

    t0_ = min(lc_['timestamp'].iloc[0], gui_['timestamp'].iloc[0])
    lt_ = (lc_['timestamp']  - t0_).dt.total_seconds().values
    gt_ = (gui_['timestamp'] - t0_).dt.total_seconds().values
    tg_ = np.arange(max(lt_[0], gt_[0]), min(lt_[-1], gt_[-1]) - 0.5 / _HZ, 1.0 / _HZ)

    mag_ = interp1d(gt_, gui_['magnitude'].values, kind='previous')(tg_)
    dir_ = interp1d(gt_, gui_['direction'].values, kind='previous')(tg_)

    df_ = pd.DataFrame({
        'magnitude':   mag_,
        'direction':   dir_,
        'measured_Fx': interp1d(lt_, lc_['Fx'].values)(tg_),
        'measured_Fy': interp1d(lt_, lc_['Fy'].values)(tg_),
        't_sec':       tg_ - tg_[0],
    })

    df_['measured_magnitude'] = np.sqrt(df_['measured_Fx']**2 + df_['measured_Fy']**2)
    df_['measured_direction'] = np.degrees(np.arctan2(df_['measured_Fy'], df_['measured_Fx']))

    chg_ = (df_['magnitude'].diff().abs() > 0.1) | (df_['direction'].diff().abs() > 0.5)
    chg_.iloc[0] = True
    df_['step'] = (chg_.cumsum() - 1).astype(int)
    df_ = df_[df_['step'] < _NSTEPS].copy()
    df_['t_in_step'] = df_['t_sec'] - df_.groupby('step')['t_sec'].transform('min')
    df_ = df_[df_['t_in_step'] <= _HOLD].copy()

    stable_ = df_[(df_['t_in_step'] >= _HOLD - _STABLE) & (df_['magnitude'] > 0)].copy()
    stable_ = stable_[stable_['magnitude'].apply(lambda m: any(abs(m - mg) < 1.0 for mg in _MAGS))]
    if stable_.empty:
        print(f'[{label}] no stable rows'); continue

    summ_ = stable_.groupby('step').agg(
        angle     = ('direction',          'first'),
        given_mag = ('magnitude',          'first'),
        mean_meas = ('measured_magnitude', 'mean'),
        mean_dir  = ('measured_direction', 'mean'),
        std_meas  = ('measured_magnitude', 'std'),
    ).reset_index()
    summ_['mean_err'] = summ_['mean_meas'] - summ_['given_mag']
    summ_['dir_err']  = summ_['mean_dir']  - summ_['angle']
    summ_['label']    = label
    summ_['pos_x']    = pos_x
    summ_['pos_z']    = pos_z
    all_summaries.append(summ_)
    print(f'[{label}]  pos=({pos_x:.3f}, {pos_z:.3f})  steps={len(summ_)}  '
          f'mean|err|={summ_["mean_err"].abs().mean():.2f}N')

combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
print(f'
Total: {len(all_summaries)} sessions loaded,  {len(combined)} step-summaries')

# save combined table
combined.to_csv(f'{BASE}/combined_errors.csv', index=False)
print(f'Saved -> {BASE}/combined_errors.csv')


In [ ]:
%matplotlib inline

if combined.empty:
    print('No data — populate SESSIONS and re-run the pipeline cell.')
else:
    loc_sum = (combined.groupby(['label', 'pos_x', 'pos_z'])
               .agg(mean_abs_err = ('mean_err', lambda x: x.abs().mean()),
                    mean_dir_err = ('dir_err',  lambda x: x.abs().mean()))
               .reset_index())

    loc_colors = {lbl: c for lbl, c in zip(loc_sum['label'], plt.cm.tab10.colors)}
    mag_levels = sorted(combined['given_mag'].unique())
    mag_colors = ['#4c9be8', '#f5a623', '#7ed321', '#e05252']

    fig = plt.figure(figsize=(18, 11), constrained_layout=True)
    gs  = gridspec.GridSpec(2, 3, figure=fig)

    # [top-left] workspace map — magnitude error
    ax_map = fig.add_subplot(gs[0, 0])
    sc = ax_map.scatter(loc_sum['pos_x'], loc_sum['pos_z'],
                        c=loc_sum['mean_abs_err'], cmap='RdYlGn_r',
                        s=300, edgecolors='black', linewidths=0.8, zorder=3,
                        vmin=0, vmax=loc_sum['mean_abs_err'].max())
    for _, r in loc_sum.iterrows():
        ax_map.annotate(r['label'], (r['pos_x'], r['pos_z']),
                        textcoords='offset points', xytext=(7, 4), fontsize=8)
    fig.colorbar(sc, ax=ax_map, label='Mean |mag error| (N)', shrink=0.8)
    ax_map.set_xlabel('x (m)'); ax_map.set_ylabel('z (m)')
    ax_map.set_title('Workspace — magnitude accuracy')
    ax_map.grid(True, alpha=0.3); ax_map.set_aspect('equal', adjustable='datalim')

    # [top-mid] workspace map — direction error
    ax_map2 = fig.add_subplot(gs[0, 1])
    sc2 = ax_map2.scatter(loc_sum['pos_x'], loc_sum['pos_z'],
                          c=loc_sum['mean_dir_err'], cmap='RdYlGn_r',
                          s=300, edgecolors='black', linewidths=0.8, zorder=3,
                          vmin=0, vmax=loc_sum['mean_dir_err'].max())
    for _, r in loc_sum.iterrows():
        ax_map2.annotate(r['label'], (r['pos_x'], r['pos_z']),
                         textcoords='offset points', xytext=(7, 4), fontsize=8)
    fig.colorbar(sc2, ax=ax_map2, label='Mean |dir error| (°)', shrink=0.8)
    ax_map2.set_xlabel('x (m)'); ax_map2.set_ylabel('z (m)')
    ax_map2.set_title('Workspace — direction accuracy')
    ax_map2.grid(True, alpha=0.3); ax_map2.set_aspect('equal', adjustable='datalim')

    # [top-right] per-location summary table
    ax_tbl = fig.add_subplot(gs[0, 2])
    ax_tbl.axis('off')
    tbl_data = [[r['label'], f'{r[pos_x]:.3f}', f'{r[pos_z]:.3f}',
                 f'{r[mean_abs_err]:.2f}', f'{r[mean_dir_err]:.1f}']
                for _, r in loc_sum.iterrows()]
    tbl = ax_tbl.table(cellText=tbl_data,
                       colLabels=['Session', 'x (m)', 'z (m)', '|Mag err| (N)', '|Dir err| (°)'],
                       loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
    tbl.scale(1.0, 1.6)
    ax_tbl.set_title('Per-location summary', fontsize=10)

    # [bottom-left] error vs magnitude box plots
    ax_mag = fig.add_subplot(gs[1, 0])
    bp = ax_mag.boxplot([combined.loc[combined['given_mag'] == m, 'mean_err'].values
                         for m in mag_levels],
                        patch_artist=True, widths=0.5,
                        medianprops=dict(color='black', lw=2))
    for patch, color in zip(bp['boxes'], mag_colors):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    ax_mag.axhline(0, color='black', lw=0.8, ls=':')
    ax_mag.set_xticks(range(1, len(mag_levels) + 1))
    ax_mag.set_xticklabels([f'{int(m)} N' for m in mag_levels])
    ax_mag.set_xlabel('Commanded magnitude'); ax_mag.set_ylabel('Magnitude error (N)')
    ax_mag.set_title('Error vs magnitude  (all locations pooled)')
    ax_mag.grid(True, axis='y', alpha=0.3)

    # [bottom-mid] magnitude error vs angle per location
    ax_ang = fig.add_subplot(gs[1, 1])
    for lbl in combined['label'].unique():
        sub = (combined[combined['label'] == lbl]
               .groupby('angle')['mean_err'].mean()
               .reset_index().sort_values('angle'))
        ax_ang.plot(sub['angle'], sub['mean_err'], marker='o', ms=5,
                    color=loc_colors[lbl], label=lbl)
    ax_ang.axhline(0, color='black', lw=0.8, ls=':')
    ax_ang.set_xlabel('Commanded angle (°)'); ax_ang.set_ylabel('Mean magnitude error (N)')
    ax_ang.set_title('Magnitude error vs angle')
    ax_ang.legend(fontsize=8); ax_ang.grid(True, alpha=0.3)

    # [bottom-right] direction error vs angle per location
    ax_dir = fig.add_subplot(gs[1, 2])
    for lbl in combined['label'].unique():
        sub = (combined[combined['label'] == lbl]
               .groupby('angle')['dir_err'].mean()
               .reset_index().sort_values('angle'))
        ax_dir.plot(sub['angle'], sub['dir_err'], marker='o', ms=5,
                    color=loc_colors[lbl], label=lbl)
    ax_dir.axhline(0, color='black', lw=0.8, ls=':')
    ax_dir.set_xlabel('Commanded angle (°)'); ax_dir.set_ylabel('Mean direction error (°)')
    ax_dir.set_title('Direction error vs angle')
    ax_dir.legend(fontsize=8); ax_dir.grid(True, alpha=0.3)

    fig.suptitle('NOARK Force Validation — Multi-location Summary', fontsize=13, fontweight='bold')
    plt.savefig(f'{BASE}/multi_location_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved -> {BASE}/multi_location_summary.png")
